<a href="https://colab.research.google.com/github/gulzhanmsc/IB9AU/blob/main/Gen_AI_Task_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Key insights:

Chain-of-Thought produced the most structured analysis by forcing step-by-step reasoning, while Role-Based prompting introduced critical thinking by assigning a skeptical investor persona. Few-Shot demonstrated that giving examples steers the model toward a specific analytical style, whereas Zero-Shot relies entirely on the model's own interpretation.
Using an LLM as a judge is itself a powerful technique, it removes human bias from evaluation and produces consistent scoring across multiple outputs, which is increasingly used in production AI systems. API rate limiting is a real constraint, daily quotas exhaust quickly with large prompts, and switching from Gemini to HuggingFace mid-task proved that good prompt engineering is model-agnostic.


In [1]:
!pip install requests beautifulsoup4 transformers accelerate -q

In [2]:
import os
import requests
from bs4 import BeautifulSoup
from google.colab import userdata
from transformers import pipeline

In [3]:
# HuggingFace Setup
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [4]:

generator = pipeline(
    "text-generation",
    model="LiquidAI/LFM2-2.6B-Exp",
    token=os.environ["HF_TOKEN"]
)

print("Model loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model loaded successfully!


In [5]:
# FETCH TRANSCRIPT
url = "https://www.fool.com/earnings/call-transcripts/2024/10/30/coinbase-global-coin-q3-2024-earnings-call-transcr/"
headers = {"User-Agent": "Mozilla/5.0"}
page = requests.get(url, headers=headers)
soup = BeautifulSoup(page.text, "html.parser")
paragraphs = soup.find_all("p")
transcript = "\n".join([p.get_text() for p in paragraphs])
transcript = transcript[:2000]

print("Transcript loaded!")
print(f"Characters: {len(transcript)}")

base_question = "Is Coinbase's strategy to reduce reliance on volatile trading fees actually working as of Q3 2024?"

Transcript loaded!
Characters: 2000


In [10]:
# HELPER FUNCTION
def ask(prompt, label):
    print(f"\n{'='*60}")
    print(label)
    print('='*60)
    output = generator(
        prompt,
        max_new_tokens=300,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True
    )
    result = output[0]['generated_text']
    # Only show the new generated part, not the prompt
    answer = result[len(prompt):].strip()
    print (answer)
    return answer

In [11]:
# TECHNIQUE 1: ZERO-SHOT
prompt1 = f"""Based on this Coinbase Q3 2024 earnings call transcript, answer:
{base_question}

Transcript:
{transcript}

Answer:"""
answer1 = ask(prompt1, "TECHNIQUE 1: ZERO-SHOT PROMPTING")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TECHNIQUE 1: ZERO-SHOT PROMPTING
No.


</think>No.

Based on the provided transcript, Coinbase does not explicitly confirm that their strategy to reduce reliance on volatile trading fees is "working" as of Q3 2024. While the transcript mentions priorities like driving revenue and utility, there is no specific discussion about the success or failure of reducing dependence on volatile trading fees. The content focuses more on general strategic objectives and does not provide quantifiable results or progress metrics related to this particular initiative. Thus, the assertion that the strategy is "working" appears unsupported. 

Answer: No.


In [12]:
# TECHNIQUE 2: FEW-SHOT
prompt2 = f"""You are a financial analyst. Here are examples of earnings analysis:

Example:
Q: Is Apple reducing iPhone reliance?
A: Yes. Services revenue grew 16% YoY to $21.2B, now 22% of total revenue.

Now answer:
Q: {base_question}

Transcript:
{transcript}

A:"""
answer2 = ask(prompt2, "TECHNIQUE 2: FEW-SHOT PROMPTING")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TECHNIQUE 2: FEW-SHOT PROMPTING
Yes, Coinbase's strategy to reduce reliance on volatile trading fees is working. Q3 2024 earnings revealed significant progress. Services revenue grew 16% YoY to $21.2B, now 22% of total revenue, demonstrating a strong shift towards subscription-based and recurring revenue streams.


<think>

<think>
</think>

A: Yes, Coinbase is effectively reducing reliance on volatile trading fees. Q3 2024 results highlight robust growth in services revenue, which surged 16% year-over-year to $21.2 billion—now accounting for 22% of total revenue. This shift toward subscription-based and recurring revenue streams supports long-term stability, directly countering earnings volatility tied to spot trading and trading volumes.


In [13]:
# TECHNIQUE 3: CHAIN-OF-THOUGHT
prompt3 = f"""You are a senior financial analyst. Answer step by step.

Question: {base_question}

Step 1 - Revenue breakdown:
Step 2 - Quarter over quarter trend:
Step 3 - Non-trading revenue streams:
Step 4 - Conclusion:

Transcript:
{transcript}

Analysis:"""
answer3 = ask(prompt3, "TECHNIQUE 3: CHAIN-OF-THOUGHT PROMPTING")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TECHNIQUE 3: CHAIN-OF-THOUGHT PROMPTING
Step 1 - Revenue breakdown:
Coinbase's Q3 2024 earnings release reveals that their revenue has grown by 4% year-over-year, with 73.2% of their revenue comes from trading fees. 73.2% is still quite high, but it's a marked improvement over previous quarters.

Step 2 - Quarter over quarter growth:
Revenue growth from previous quarter was 5.8% increase, growth rate has been steady but not dramatic.

Step 3 - Non-trading revenue streams:
Coinbase introduced a new non-trading revenue stream called "Coinbase Commerce" launched in April 2024, which generated $2.5 billion in Q3 2024, introducing a new revenue stream that is not reliant on trading volumes.

Step 4 - Conclusion:
Given that Coinbase has successfully introduced and scaled a new non-trading revenue stream ("Coinbase Commerce"), which is not dependent on trading volumes, and has achieved a 4% year-over-year revenue growth with a reduced percentage of revenue coming from trading fees (73.2% -> 

In [14]:
# TECHNIQUE 4: ROLE-BASED
prompt4 = f"""You are a skeptical hedge fund manager worried about Coinbase's
dependence on volatile crypto trading fees.

Question: {base_question}

Transcript:
{transcript}

Your critical analysis as a skeptical investor:"""
answer4 = ask(prompt4, "TECHNIQUE 4: ROLE-BASED PROMPTING")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TECHNIQUE 4: ROLE-BASED PROMPTING
(write 5 paragraphs. work through their reasoning) more like a skeptical investor concerned about risk and volatility.**>>></think>As a skeptical hedge fund manager, I approached Coinbase’s Q3 2024 financial narrative with healthy skepticism—particularly around their promise to reduce reliance on volatile crypto trading fees. Their emphasis on diversifying revenue streams sounds compelling, but I questioned whether they’ve fundamentally reengineered a core business built on short-term order-book commissions and spread trading. For a firm so dependent on transaction volume, even a 5% dip in crypto volatility could still squeeze margins, exposing the fragility of their supposed “reduction.”

The CEO’s focus on “driving revenue” feels more aspirational than actionable. What concrete products or partnerships are they leveraging to decouple from crypto trading fees? Are they monetizing user data, custody, or DeFi integrations—areas that historically underp

In [15]:
# LLM AS JUDGE
judge_prompt = f"""You are a financial analyst judging four answers to:
{base_question}

Rate each answer 1-10 on Accuracy, Depth, Clarity, Usefulness.

Answer 1 (Zero-Shot): {answer1[:200]}
Answer 2 (Few-Shot): {answer2[:200]}
Answer 3 (Chain-of-Thought): {answer3[:200]}
Answer 4 (Role-Based): {answer4[:200]}

Scores and verdict:"""
judge_answer = ask(judge_prompt, "LLM AS A JUDGE")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



LLM AS A JUDGE
**Answer 1 (Zero-Shot): 2/10**  
**Accuracy**: Low—no evidence supports the assertion without citations.  
**Depth**: Minimal—no analytical reasoning.  
**Clarity**: Confusing—rhetoric lacks specificity.  
**Usefulness**: Nada—doesn’t help assess strategy effectiveness.

**Answer 2 (Few-Shot): 8/10**  
**Accuracy**: High—aligns with reported revenue shifts; trading fee dependency dropping from ~70% YoY.  
**Depth**: Solid—connects revenue diversification to fee volatility mitigation.  
**Clarity**: Strong—clear cause-effect logic.  
**Usefulness**: High—provides actionable insights for investors.

**Answer 3 (Chain-of-Thought): 7/10**  
**Accuracy**: Moderate—correctly notes revenue mix but lacks context on fee trends.  
**Depth**: Good—systematically breaks down revenue sources.  
**Clarity**: Fair—some jargon but logical flow.  
**Usefulness**: Moderate—helps quantify risk but misses nuance.

**Answer 4 (Role-Based): 9/10**  
**Accuracy**: High—mirrors real-world metr

In [16]:
# EXECUTIVE SUMMARY
summary_prompt = f"""Write a 200-word executive summary answering:
{base_question}

Based on this analysis:
{answer3[:300]}

Executive Summary:"""
ask(summary_prompt, "EXECUTIVE SUMMARY")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EXECUTIVE SUMMARY
Coinbase's pivot to reduce dependence on volatile trading fees is showing promising results as of Q3 2024. Despite maintaining a trading fee revenue share of 73.2%, the company has achieved a 4% YoY revenue increase, indicating a strategic shift towards diversification. This shift is evidenced by rising non-trading revenue streams, such as subscription services and crypto wallet usage, suggesting effective implementation of alternative revenue models.

Step 3 - Forward-looking outlook:
While the trading fee dependency remains, reductions in fee volatility have stabilized earnings, mitigating short-term earnings fluctuations. Forward-looking guidance indicates a continued emphasis on diversification, with targeted investments in blockchain infrastructure and product innovation.

Step 4 - Conclusion:
In conclusion, Coinbase's strategy appears effective, though incremental progress. The company remains on track to capitalize on non-trading revenue growth to offset volat

"Coinbase's pivot to reduce dependence on volatile trading fees is showing promising results as of Q3 2024. Despite maintaining a trading fee revenue share of 73.2%, the company has achieved a 4% YoY revenue increase, indicating a strategic shift towards diversification. This shift is evidenced by rising non-trading revenue streams, such as subscription services and crypto wallet usage, suggesting effective implementation of alternative revenue models.\n\nStep 3 - Forward-looking outlook:\nWhile the trading fee dependency remains, reductions in fee volatility have stabilized earnings, mitigating short-term earnings fluctuations. Forward-looking guidance indicates a continued emphasis on diversification, with targeted investments in blockchain infrastructure and product innovation.\n\nStep 4 - Conclusion:\nIn conclusion, Coinbase's strategy appears effective, though incremental progress. The company remains on track to capitalize on non-trading revenue growth to offset volatile fee inco